# 🔍 DBReadAgent — Local Ollama + Gradio

**Run this notebook inside VSCode with the `dbreadagent` conda kernel.**

> All dependencies are local. No API keys needed. Ollama must be running with your model pulled.

| Step | Action |
|---|---|
| 1 | Open terminal → `ollama serve` (if not already running) |
| 2 | Select kernel `Python (dbreadagent)` in VSCode |
| 3 | Run cells top-to-bottom |
| 4 | Click the Gradio link that appears in the last cell |

## ⚙️ 1. Config Check

In [38]:
# cell 1 — config (no MODEL/BASE_URL needed anymore)
import sys, os
from pathlib import Path

SRC = Path("src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv()

DB_DIR = "./databases"
print(f"🗄️  DB dir  : {Path(DB_DIR).resolve()}")
print(f"🐍  Python : {sys.version.split()[0]}")

🗄️  DB dir  : D:\_Python\02_AL_ML\02_agents\01_LangChain\databases
🐍  Python : 3.10.20


In [40]:
# cell 1b — provider picker widget (run once; re-run to switch)
import ipywidgets as widgets
from IPython.display import display

provider_widget = widgets.ToggleButtons(
    options=["ollama", "groq", "openrouter", "llamacpp"],
    value=os.getenv("LLM_PROVIDER", "groq"),
    description="Provider:",
    button_style="info",
)
model_widget = widgets.Text(
    value="",
    placeholder="leave blank → use provider default",
    description="Model:",
    layout=widgets.Layout(width="420px"),
)

display(provider_widget, model_widget)

ToggleButtons(button_style='info', description='Provider:', index=1, options=('ollama', 'groq', 'openrouter', …

Text(value='', description='Model:', layout=Layout(width='420px'), placeholder='leave blank → use provider def…

## 🏓 2. Ping Ollama

In [ ]:
# import urllib.request, json

# try:
#     resp = urllib.request.urlopen(f"{BASE_URL}/api/tags", timeout=4)
#     data = json.loads(resp.read())
#     models = [m["name"] for m in data.get("models", [])]
#     print(f"✅ Ollama is running. Available models:")
#     for m in models:
#         marker = " ← selected" if MODEL in m else ""
#         print(f"   • {m}{marker}")
#     if not any(MODEL in m for m in models):
#         print(f"\n⚠️  Model '{MODEL}' not found. Pull it with:")
#         print(f"     ollama pull {MODEL}")
# except Exception as e:
#     print(f"❌ Cannot reach Ollama at {BASE_URL}: {e}")
#     print("   Make sure 'ollama serve' is running in a terminal.")

✅ Ollama is running. Available models:
   • gpt-oss:120b-cloud ← selected
   • minimax-m3:cloud
   • glm-5.2:cloud
   • minimax-m2.5:cloud


## 🗄️ 3. Initialise & Seed Databases

In [32]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(name)s  %(message)s")

from db_setup import init_all, get_schema_text

# init_all()   # idempotent — skips if already seeded
# print("\n✅ Databases ready.")

## 🔬 4. Inspect Schema (optional)

In [33]:
# Change 'company' to 'analytics' or 'inventory' to inspect other DBs
print(get_schema_text(["company"]))

=== DATABASE: COMPANY ===

-- assignments (116 rows) | columns: assign_id, emp_id, proj_id, role_on_proj, hours_per_week, assigned_on
CREATE TABLE assignments (assign_id INTEGER PRIMARY KEY AUTOINCREMENT, emp_id INTEGER REFERENCES employees(emp_id) ON DELETE CASCADE, proj_id INTEGER REFERENCES projects(proj_id) ON DELETE CASCADE, role_on_proj TEXT, hours_per_week INTEGER DEFAULT 10, assigned_on TEXT DEFAULT (date('now')), UNIQUE(emp_id, proj_id));

-- departments (8 rows) | columns: dept_id, dept_name, location, budget, created_at
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY AUTOINCREMENT, dept_name TEXT NOT NULL UNIQUE, location TEXT, budget REAL DEFAULT 0, created_at TEXT DEFAULT (datetime('now')));

-- employees (80 rows) | columns: emp_id, name, email, role, salary, dept_id, hire_date, level, is_active
CREATE TABLE employees (emp_id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE, role TEXT, salary REAL, dept_id INTEGER REFERENCES departme

## 🤖 5. Build Agent

In [ ]:
# print(MODEL)

gpt-oss:120b-cloud


In [41]:
# from agent import ConversationManager

# conv = ConversationManager()
# print(f"✅ Agent ready — model: {MODEL}")

# cell 5 — build agent from widget selection
import importlib, agent
importlib.reload(agent)
from agent import ConversationManager

selected_provider = provider_widget.value
selected_model    = model_widget.value.strip() or None

if selected_model:
    os.environ["LLM_MODEL"] = selected_model
elif "LLM_MODEL" in os.environ:
    del os.environ["LLM_MODEL"]          # fall back to provider default

conv = ConversationManager(provider=selected_provider)
print(f"✅ Agent ready — provider: {selected_provider}  model: {os.getenv('LLM_MODEL', 'default')}")

INFO  agent  LLM provider=groq  model=llama-3.3-70b-versatile  base_url=https://api.groq.com/openai/v1
INFO  agent  DBReadAgent compiled  provider=groq


✅ Agent ready — provider: groq  model: default


## 💬 6. Quick In-Notebook Test

In [42]:
import importlib, os

import agent
importlib.reload(agent)

<module 'agent' from 'd:\\_Python\\02_AL_ML\\02_agents\\01_LangChain\\agent.py'>

In [43]:
# Simple test
answer, sql = conv.ask("List all departments with employee headcount and average salary.")
print(answer)
if sql:
    print("\n── SQL used ──")
    for s in sql: print(s)

INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO  openai._base_client  Retrying request to /chat/completions in 25.000000 seconds
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


There are 8 departments with the following headcounts and average salaries:
- Department 2: 14 employees, $140,707.14 average salary
- Department 7: 14 employees, $127,478.57 average salary
- Department 4: 10 employees, $133,140.00 average salary
- Department 6: 10 employees, $115,490.00 average salary
- Department 5: 9 employees, $115,166.67 average salary
- Department 1: 8 employees, $108,562.50 average salary
- Department 8: 8 employees, $141,262.50 average salary
- Department 3: 7 employees, $154,471.43 average salary

The total number of employees is 80.

── SQL used ──
SELECT dept_id, COUNT(*) AS headcount, AVG(salary) AS avg_salary FROM employees GROUP BY dept_id ORDER BY headcount DESC LIMIT 100
SELECT COUNT(*) FROM employees LIMIT 500


In [44]:
# Follow-up — tests conversation memory
answer2, sql2 = conv.ask("Now break the highest-budget department down by seniority level.")
print(answer2)

INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO  openai._base_client  Retrying request to /chat/completions in 9.000000 seconds
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO  httpx  HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO  httpx  HTTP 

The highest-budget department is Department 3, with a total of 7 employees and an average salary of $154,471.43. The salaries in this department range from $109,400 to $235,100.


## 🚀 7. Launch Gradio UI

In [ ]:
# ── Option A: Launch inline (Jupyter) ────────────────────────────────────────
from app import build_ui
import os

PORT  = int(os.getenv("GRADIO_SERVER_PORT", 7860))
HOST  = os.getenv("GRADIO_SERVER_NAME", "127.0.0.1")
SHARE = os.getenv("GRADIO_SHARE", "false").lower() == "true"

ui = build_ui()
ui.launch(server_name=HOST, server_port=PORT, share=SHARE, show_error=True)

# ── Option B: Run as standalone script (from terminal) ────────────────────────
# cd dbreadagent
# python src/app.py

d:\_Python\02_AL_ML\02_agents\01_LangChain\app.py:199: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="DBReadAgent") as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\aviav.NITRO-5\miniconda3\envs\agentAPI\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\aviav.NITRO-5\miniconda3\envs\agentAPI\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\aviav.NITRO-5\miniconda3\envs\agentAPI\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\aviav.NITRO-5\miniconda3\envs\agentAPI\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_